In [ ]:
# Cell 0 - Clean environment setup
import os

# 1. Pin numpy to 1.x to prevent C-header binary incompatibility
!pip install "numpy<2.0" -q

# 2. Install unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# 3. Install dependencies to satisfy ALL broken internal callbacks
!pip install "trl>=0.18.2,<=0.24.0" mergekit llm-blender -q
!pip install "datasets==3.6.0" -q

# 4. FORCE RESTART the Colab runtime to clear the old NumPy from memory
print("Installing complete. Restarting runtime to apply changes...")
os.kill(os.getpid(), 9)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires click!=8.2.*,>=4.0, but you have click 8.2.1 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
bigframes 2.37.0 requires rich<14,>=12.4.4, but you have rich 14.3.3 which is incompatible.
mcp 1.26.0 requires pydantic<3.0.0,>=2.11.0, but you have pydantic 2.10.6 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.


In [ ]:
# # Cell 2 - Generate pairs
# import os
# os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# from unsloth import FastLanguageModel
# from google.colab import drive
# from datasets import load_dataset
# from peft import PeftModel
# import torch, json, random
# from tqdm import tqdm

# drive.mount('/content/drive')

# # Load fine-tuned teacher
# print("Loading fine-tuned model...")
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/meta-llama-3.1-8b-bnb-4bit",
#     max_seq_length = 1024,
#     load_in_4bit = True,
# )
# model = PeftModel.from_pretrained(
#     model,
#     "/content/drive/MyDrive/therapy_teacher_lora"
# )
# FastLanguageModel.for_inference(model)
# print("Model loaded!")

# # Load dataset and sample 500 prompts
# dataset = load_dataset(
#     "json",
#     data_files = "/content/therapy_dataset_final.jsonl",
#     split = "train"
# )
# random.seed(42)
# indices = random.sample(range(len(dataset)), 500)
# sampled = dataset.select(indices)

# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# {}

# ### Input:
# {}

# ### Response:
# """

# def generate_response(prompt, temperature):
#     inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens = 150,
#             use_cache = True,
#             temperature = temperature,
#             do_sample = True,
#         )
#     response = tokenizer.batch_decode(outputs)[0]
#     response = response.split("### Response:")[-1].strip()
#     response = response.replace("<|end_of_text|>", "").strip()
#     return response

# # Generate two responses per prompt
# print("Generating response pairs (this takes ~30 mins)...")
# pairs = []

# for i, example in enumerate(tqdm(sampled)):
#     prompt = alpaca_prompt.format(
#         example["instruction"],
#         example["input"]
#     )

#     # Response A — more conservative
#     response_a = generate_response(prompt, temperature=0.3)
#     # Response B — more creative
#     response_b = generate_response(prompt, temperature=0.9)

#     pairs.append({
#         "instruction": example["instruction"],
#         "input": example["input"],
#         "response_a": response_a,
#         "response_b": response_b,
#     })

#     # Save checkpoint every 50
#     if (i + 1) % 50 == 0:
#         with open("/content/drive/MyDrive/response_pairs.jsonl", "w") as f:
#             for item in pairs:
#                 f.write(json.dumps(item) + "\n")
#         print(f"Checkpoint: {i+1}/500 saved")

# # Final save
# with open("/content/drive/MyDrive/response_pairs.jsonl", "w") as f:
#     for item in pairs:
#         f.write(json.dumps(item) + "\n")

# print(f"Done! {len(pairs)} pairs saved.")

In [ ]:
#  # Cell 3 - Judge Agent
# from groq import Groq
# import json, time
# from tqdm import tqdm

# GROQ_API_KEY = ""  # paste your key here
# client = Groq(api_key=GROQ_API_KEY)

# JUDGE_PROMPT = """You are an expert clinical psychologist evaluating AI therapy responses.

# You will be given a patient message and two responses (A and B).
# Evaluate which response is better based on these criteria:
# 1. EMPATHY: Does it acknowledge the patient's feelings warmly?
# 2. SAFETY: Does it avoid prescribing medication or making diagnoses?
# 3. CLARITY: Is it clear and not repetitive?
# 4. PROFESSIONALISM: Does it sound like a licensed counselor?

# Patient message: {input}

# Response A: {response_a}

# Response B: {response_b}

# Reply with ONLY one of these two words: CHOOSE_A or CHOOSE_B"""

# def judge_responses(input_text, response_a, response_b):
#     try:
#         result = client.chat.completions.create(
#             model = "llama-3.3-70b-versatile",
#             messages = [
#                 {
#                     "role": "user",
#                     "content": JUDGE_PROMPT.format(
#                         input = input_text,
#                         response_a = response_a,
#                         response_b = response_b
#                     )
#                 }
#             ],
#             max_tokens = 10,
#             temperature = 0.0,
#         )
#         verdict = result.choices[0].message.content.strip()
#         return verdict
#     except Exception as e:
#         print(f"Judge error: {e}")
#         time.sleep(5)
#         return "CHOOSE_A"  # default to A on error

# # Load pairs
# pairs = []
# with open("/content/drive/MyDrive/response_pairs.jsonl", "r") as f:
#     for line in f:
#         pairs.append(json.loads(line))

# # Run judge on all pairs
# print("Running Judge Agent...")
# dpo_dataset = []

# for i, pair in enumerate(tqdm(pairs)):
#     verdict = judge_responses(
#         pair["input"],
#         pair["response_a"],
#         pair["response_b"]
#     )

#     if "CHOOSE_A" in verdict:
#         chosen   = pair["response_a"]
#         rejected = pair["response_b"]
#     else:
#         chosen   = pair["response_b"]
#         rejected = pair["response_a"]

#     dpo_dataset.append({
#         "prompt": pair["instruction"] + "\n" + pair["input"],
#         "chosen": chosen,
#         "rejected": rejected,
#     })

#     # Groq rate limit — small delay
#     time.sleep(0.5)

#     # Save checkpoint every 50
#     if (i + 1) % 50 == 0:
#         with open("/content/drive/MyDrive/dpo_dataset.jsonl", "w") as f:
#             for item in dpo_dataset:
#                 f.write(json.dumps(item) + "\n")
#         print(f"Checkpoint: {i+1}/500 judged")

# # Final save
# with open("/content/drive/MyDrive/dpo_dataset.jsonl", "w") as f:
#     for item in dpo_dataset:
#         f.write(json.dumps(item) + "\n")

# print(f"DPO dataset ready with {len(dpo_dataset)} preference pairs!")

In [ ]:
# # Cell 3 - Judge Agent (with resume + rate limit handling)
# from groq import Groq
# import json, time, re, os
# from tqdm import tqdm

# GROQ_API_KEY = ""
# client = Groq(api_key=GROQ_API_KEY)

# JUDGE_PROMPT = """You are an expert clinical psychologist evaluating AI therapy responses.

# You will be given a patient message and two responses (A and B).
# Evaluate which response is better based on these criteria:
# 1. EMPATHY: Does it acknowledge the patient's feelings warmly?
# 2. SAFETY: Does it avoid prescribing medication or making diagnoses?
# 3. CLARITY: Is it clear and not repetitive?
# 4. PROFESSIONALISM: Does it sound like a licensed counselor?

# Patient message: {input}

# Response A: {response_a}

# Response B: {response_b}

# Reply with ONLY one of these two words: CHOOSE_A or CHOOSE_B"""

# def parse_wait_seconds(error_message):
#     """Extract wait time from Groq rate limit error message."""
#     match = re.search(r'try again in (\d+)m([\d.]+)s', str(error_message))
#     if match:
#         return int(match.group(1)) * 60 + float(match.group(2)) + 5  # +5s buffer
#     return 300  # default 5 min if can't parse

# def judge_responses(input_text, response_a, response_b, retries=3):
#     for attempt in range(retries):
#         try:
#             result = client.chat.completions.create(
#                 model="llama-3.3-70b-versatile",
#                 messages=[{
#                     "role": "user",
#                     "content": JUDGE_PROMPT.format(
#                         input=input_text,
#                         response_a=response_a,
#                         response_b=response_b
#                     )
#                 }],
#                 max_tokens=10,
#                 temperature=0.0,
#             )
#             verdict = result.choices[0].message.content.strip()
#             if "CHOOSE_A" in verdict or "CHOOSE_B" in verdict:
#                 return verdict
#             return None
#         except Exception as e:
#             err_str = str(e)
#             if "rate_limit_exceeded" in err_str and "tokens per day" in err_str:
#                 wait_secs = parse_wait_seconds(err_str)
#                 print(f"\n⏳ Daily TPD limit hit. Waiting {wait_secs:.0f}s (~{wait_secs/60:.1f} min)...")
#                 time.sleep(wait_secs)
#                 # Don't count this as a retry — loop will retry automatically
#             elif "rate_limit_exceeded" in err_str:
#                 print(f"Rate limit (TPM). Waiting 60s...")
#                 time.sleep(60)
#             else:
#                 print(f"Attempt {attempt+1} failed: {e}")
#                 time.sleep(5)
#     return None

# # Load all pairs
# pairs = []
# with open("/content/drive/MyDrive/response_pairs.jsonl", "r") as f:
#     for line in f:
#         pairs.append(json.loads(line))

# # ✅ RESUME: Load existing checkpoint if it exists
# dpo_dataset = []
# checkpoint_path = "/content/drive/MyDrive/dpo_dataset.jsonl"
# start_index = 0

# if os.path.exists(checkpoint_path):
#     with open(checkpoint_path, "r") as f:
#         for line in f:
#             dpo_dataset.append(json.loads(line))
#     start_index = len(dpo_dataset)
#     print(f"Resuming from pair {start_index}/500 ({len(dpo_dataset)} already judged)")
# else:
#     print("Starting fresh...")

# # Run judge — only on remaining pairs
# print("Running Judge Agent...")
# remaining_pairs = pairs[start_index:]

# for i, pair in enumerate(tqdm(remaining_pairs, initial=start_index, total=len(pairs))):
#     verdict = judge_responses(pair["input"], pair["response_a"], pair["response_b"])

#     if verdict is None:
#         print(f"Skipping pair {start_index + i} — judge failed")
#         continue

#     dpo_dataset.append({
#         "prompt": pair["instruction"] + "\n" + pair["input"],
#         "chosen": pair["response_a"] if "CHOOSE_A" in verdict else pair["response_b"],
#         "rejected": pair["response_b"] if "CHOOSE_A" in verdict else pair["response_a"],
#     })

#     time.sleep(0.5)  # small delay to avoid TPM limits

#     if (start_index + i + 1) % 50 == 0:
#         with open(checkpoint_path, "w") as f:
#             for item in dpo_dataset:
#                 f.write(json.dumps(item) + "\n")
#         print(f"Checkpoint: {start_index + i + 1}/500 judged")

# # Final save
# with open(checkpoint_path, "w") as f:
#     for item in dpo_dataset:
#         f.write(json.dumps(item) + "\n")

# print(f"✅ DPO dataset ready with {len(dpo_dataset)} preference pairs!")

In [ ]:
# Cell 4 - DPO Training (clean)
import gc, torch

gc.collect()
torch.cuda.empty_cache()

from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
from peft import PeftModel

PatchDPOTrainer()

print("Loading model for DPO...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-llama-3.1-8b-bnb-4bit",
    max_seq_length = 1024,
    load_in_4bit = True,
)
model = PeftModel.from_pretrained(
    model,
    "/content/drive/MyDrive/therapy_teacher_lora"
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

dpo_data = load_dataset(
    "json",
    data_files = "/content/drive/MyDrive/dpo_dataset.jsonl",
    split = "train"
)
print(f"DPO dataset size: {len(dpo_data)}")

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    tokenizer = tokenizer,
    train_dataset = dpo_data,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 5e-5,
        fp16 = True,
        logging_steps = 10,
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = "dpo_outputs",
        beta = 0.1,
        max_length = 1024,
        max_prompt_length = 512,
    ),
)

print("Starting DPO training...")
dpo_trainer.train()

model.save_pretrained("/content/drive/MyDrive/therapy_dpo_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/therapy_dpo_lora")
print("DPO model saved!")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


RuntimeError: Failed to import trl.trainer.dpo_trainer because of the following error (look up to see its traceback):
No module named 'llm_blender'